# Final Experiments: Variance, F1, and Architecture Comparison
## Closing Dissertation Gaps Across SHD, ECG, and FI-2010

**Purpose**: This notebook addresses the methodological gaps identified in the experiment review:
1. **Multi-seed variance** (5 seeds) on key configurations across all 3 datasets
2. **F1 scores** (macro and weighted) reported consistently alongside accuracy
3. **B0–B7 architecture comparison** across all 3 datasets (completing the timed-out master notebooks)
4. **Per-class metrics** and confusion matrices for imbalanced datasets (ECG, FI-2010)

**Data splits are IDENTICAL to the existing notebooks**:
- SHD: 8,156 train / 2,264 test, 100 time bins, 700 channels
- ECG: Strodthoff folds 1–8 train / fold 9 val / fold 10 test, 250 bins, delta encoding
- FI-2010: Days 1–7 train (80/20 val), Days 8–9 test (139,488 samples), k=50 horizon

**Estimated runtime**: ~18–24 hours on T4 (5 seeds × 3 datasets × 9 architectures).

## 0. Setup

In [1]:
import sys, subprocess
subprocess.check_call([sys.executable, '-m', 'pip', 'install', 'snntorch', 'wfdb', '-q'])
print('Done.')

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 125.6/125.6 kB 4.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 163.9/163.9 kB 8.8 MB/s eta 0:00:00


Done.


In [2]:
import os, glob, copy, time, json, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from sklearn.metrics import (classification_report, confusion_matrix,
                             f1_score, accuracy_score)
warnings.filterwarnings('ignore')

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {device}')
if device.type == 'cuda':
    print(f'GPU: {torch.cuda.get_device_name()}')

N_SEEDS = 5
SEEDS = [42, 123, 456, 789, 1024]
SAVE_DIR = '/kaggle/working'
ALL_RESULTS = {}

def set_seed(seed):
    torch.manual_seed(seed)
    np.random.seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

set_seed(42)
print(f'Running {N_SEEDS} seeds: {SEEDS}')

Device: cuda
GPU: Tesla T4
Running 5 seeds: [42, 123, 456, 789, 1024]


## 1. Spike Encoding

In [3]:
class SpikeEncoder:
    @staticmethod
    def direct(signal):
        return signal

    @staticmethod
    def rate_coding(signal):
        sig_min = signal.min(axis=0, keepdims=True)
        sig_max = signal.max(axis=0, keepdims=True)
        probs = (signal - sig_min) / (sig_max - sig_min + 1e-8)
        return (np.random.rand(*signal.shape) < probs).astype(np.float32)

    @staticmethod
    def delta_modulation(signal, threshold=0.1):
        T, C = signal.shape
        spikes = np.zeros_like(signal)
        reference = signal[0].copy()
        for t in range(1, T):
            diff = signal[t] - reference
            up = diff > threshold
            down = diff < -threshold
            spikes[t, up] = 1.0
            spikes[t, down] = -1.0
            reference[up] = signal[t, up]
            reference[down] = signal[t, down]
        return spikes

    @staticmethod
    def adaptive_delta(signal, percentile=95):
        T, C = signal.shape
        diffs = np.abs(np.diff(signal, axis=0))
        thresholds = np.percentile(diffs, percentile, axis=0)
        thresholds = np.maximum(thresholds, 1e-8)
        spikes = np.zeros_like(signal)
        reference = signal[0].copy()
        for t in range(1, T):
            diff = signal[t] - reference
            up = diff > thresholds
            down = diff < -thresholds
            spikes[t, up] = 1.0
            spikes[t, down] = -1.0
            reference[up] = signal[t, up]
            reference[down] = signal[t, down]
        return spikes, thresholds

## 2. SNN Components

All components identical to the existing notebooks:
- Surrogate gradient: fast sigmoid, beta=40 (Cramer et al. 2020 Eq. 8)
- LIF layer with current-based synapses (Eqs. 5–6)
- Spike regularisation (Eqs. 10–11)

In [4]:
class SurrogateSpike(torch.autograd.Function):
    beta = 40.0
    @staticmethod
    def forward(ctx, mem, threshold=1.0):
        ctx.save_for_backward(mem)
        ctx.threshold = threshold
        return (mem >= threshold).float()
    @staticmethod
    def backward(ctx, grad_output):
        mem, = ctx.saved_tensors
        v = mem - ctx.threshold
        grad = 1.0 / (1.0 + SurrogateSpike.beta * torch.abs(v)) ** 2
        return grad_output * grad, None

def spike_fn(x, threshold=1.0):
    return SurrogateSpike.apply(x, threshold)

In [5]:
class LIFLayer(nn.Module):
    def __init__(self, input_size, hidden_size, recurrent=False,
                 tau_mem_init=20.0, tau_syn_init=10.0, dt=10.0,
                 learnable_tau=False, dropout=0.0,
                 heterogeneous_tau=False):
        super().__init__()
        self.hidden_size = hidden_size
        self.recurrent = recurrent
        self.dt = dt
        self.dropout = dropout
        self.W_ff = nn.Linear(input_size, hidden_size, bias=False)
        if recurrent:
            self.W_rec = nn.Linear(hidden_size, hidden_size, bias=False)
        if heterogeneous_tau:
            log_tau_mem = torch.empty(hidden_size).uniform_(np.log(5.0), np.log(200.0))
            log_tau_syn = torch.empty(hidden_size).uniform_(np.log(2.0), np.log(100.0))
            self.log_tau_mem = nn.Parameter(log_tau_mem)
            self.log_tau_syn = nn.Parameter(log_tau_syn)
        elif learnable_tau:
            self.log_tau_mem = nn.Parameter(torch.tensor(np.log(tau_mem_init)))
            self.log_tau_syn = nn.Parameter(torch.tensor(np.log(tau_syn_init)))
        else:
            self.register_buffer('log_tau_mem', torch.tensor(np.log(tau_mem_init)))
            self.register_buffer('log_tau_syn', torch.tensor(np.log(tau_syn_init)))
        nn.init.kaiming_uniform_(self.W_ff.weight, nonlinearity='linear')
        if recurrent:
            nn.init.kaiming_uniform_(self.W_rec.weight, nonlinearity='linear')

    @property
    def alpha(self):
        return torch.exp(-self.dt / torch.exp(self.log_tau_syn))
    @property
    def beta_decay(self):
        return torch.exp(-self.dt / torch.exp(self.log_tau_mem))

    def forward(self, x):
        B, T, _ = x.shape
        alpha, beta = self.alpha, self.beta_decay
        syn = torch.zeros(B, self.hidden_size, device=x.device)
        mem = torch.zeros(B, self.hidden_size, device=x.device)
        prev_spk = torch.zeros(B, self.hidden_size, device=x.device)
        spk_rec, mem_rec = [], []
        for t in range(T):
            cur = self.W_ff(x[:, t])
            syn = alpha * syn + cur
            if self.recurrent:
                rec_in = F.dropout(prev_spk, p=self.dropout, training=self.training) if self.dropout > 0 else prev_spk
                syn = syn + self.W_rec(rec_in)
            mem = beta * mem * (1.0 - prev_spk) + (1.0 - beta) * syn
            spk = spike_fn(mem)
            spk_rec.append(spk)
            mem_rec.append(mem)
            prev_spk = spk
        return torch.stack(spk_rec, dim=1), torch.stack(mem_rec, dim=1)


class ALIFLayer(nn.Module):
    """Adaptive LIF. Bellec et al. 2020, Nature Comms."""
    def __init__(self, input_size, hidden_size, recurrent=False,
                 tau_mem_init=20.0, tau_syn_init=10.0, dt=10.0,
                 tau_adapt=100.0, beta_adapt=0.1,
                 learnable_tau=False, dropout=0.0,
                 heterogeneous_tau=False):
        super().__init__()
        self.hidden_size = hidden_size
        self.recurrent = recurrent
        self.dt = dt
        self.dropout = dropout
        self.beta_adapt = beta_adapt
        self.rho = np.exp(-dt / tau_adapt)
        self.W_ff = nn.Linear(input_size, hidden_size, bias=False)
        if recurrent:
            self.W_rec = nn.Linear(hidden_size, hidden_size, bias=False)
        if learnable_tau:
            self.log_tau_mem = nn.Parameter(torch.tensor(np.log(tau_mem_init)))
            self.log_tau_syn = nn.Parameter(torch.tensor(np.log(tau_syn_init)))
        else:
            self.register_buffer('log_tau_mem', torch.tensor(np.log(tau_mem_init)))
            self.register_buffer('log_tau_syn', torch.tensor(np.log(tau_syn_init)))
        nn.init.kaiming_uniform_(self.W_ff.weight, nonlinearity='linear')
        if recurrent:
            nn.init.kaiming_uniform_(self.W_rec.weight, nonlinearity='linear')

    @property
    def alpha(self):
        return torch.exp(-self.dt / torch.exp(self.log_tau_syn))
    @property
    def beta_decay(self):
        return torch.exp(-self.dt / torch.exp(self.log_tau_mem))

    def forward(self, x):
        B, T, _ = x.shape
        alpha, beta = self.alpha, self.beta_decay
        syn = torch.zeros(B, self.hidden_size, device=x.device)
        mem = torch.zeros(B, self.hidden_size, device=x.device)
        prev_spk = torch.zeros(B, self.hidden_size, device=x.device)
        adapt = torch.zeros(B, self.hidden_size, device=x.device)
        spk_rec, mem_rec = [], []
        for t in range(T):
            cur = self.W_ff(x[:, t])
            syn = alpha * syn + cur
            if self.recurrent:
                rec_in = F.dropout(prev_spk, p=self.dropout, training=self.training) if self.dropout > 0 else prev_spk
                syn = syn + self.W_rec(rec_in)
            adapt = self.rho * adapt + prev_spk
            thr = 1.0 + self.beta_adapt * adapt
            mem = beta * mem * (1.0 - prev_spk) + (1.0 - beta) * syn
            spk = spike_fn(mem, thr)
            spk_rec.append(spk)
            mem_rec.append(mem)
            prev_spk = spk
        return torch.stack(spk_rec, dim=1), torch.stack(mem_rec, dim=1)

In [6]:
class ReadoutLayer(nn.Module):
    def __init__(self, input_size, output_size, tau_mem=20.0, dt=10.0):
        super().__init__()
        self.fc = nn.Linear(input_size, output_size, bias=False)
        self.beta = np.exp(-dt / tau_mem)
        nn.init.kaiming_uniform_(self.fc.weight, nonlinearity='linear')
    def forward(self, x):
        B, T, _ = x.shape
        mem = torch.zeros(B, self.fc.out_features, device=x.device)
        mem_rec = []
        for t in range(T):
            mem = self.beta * mem + (1.0 - self.beta) * self.fc(x[:, t])
            mem_rec.append(mem)
        return torch.stack(mem_rec, dim=1)


class AttentionReadout(nn.Module):
    """Attention-weighted readout. Yao et al. 2023."""
    def __init__(self, input_size, output_size):
        super().__init__()
        self.fc = nn.Linear(input_size, output_size, bias=False)
        self.attn = nn.Linear(input_size, 1, bias=False)
        nn.init.kaiming_uniform_(self.fc.weight, nonlinearity='linear')
    def forward(self, x):
        B, T, _ = x.shape
        logits_seq = self.fc(x)
        weights = torch.softmax(self.attn(x).squeeze(-1), dim=1)
        output = (logits_seq * weights.unsqueeze(-1)).sum(dim=1)
        return logits_seq, output


class SNN(nn.Module):
    def __init__(self, input_size, hidden_size=256, output_size=5,
                 n_hidden_layers=1, recurrent=True,
                 tau_mem=20.0, tau_syn=10.0, dt=10.0,
                 learnable_tau=True, dropout=0.3,
                 neuron_type='lif', heterogeneous_tau=False,
                 input_bn=False, readout_type='max_over_time',
                 tau_adapt=100.0, beta_adapt=0.1):
        super().__init__()
        self.readout_type = readout_type
        self.hidden_size = hidden_size
        self.input_bn_flag = input_bn
        if input_bn:
            self.input_bn = nn.BatchNorm1d(input_size)

        LayerClass = ALIFLayer if neuron_type == 'alif' else LIFLayer
        layer_kwargs = dict(recurrent=recurrent, tau_mem_init=tau_mem, tau_syn_init=tau_syn,
                           dt=dt, learnable_tau=learnable_tau, dropout=dropout,
                           heterogeneous_tau=heterogeneous_tau)
        if neuron_type == 'alif':
            layer_kwargs['tau_adapt'] = tau_adapt
            layer_kwargs['beta_adapt'] = beta_adapt

        layers = []
        for i in range(n_hidden_layers):
            in_sz = input_size if i == 0 else hidden_size
            layers.append(LayerClass(in_sz, hidden_size, **layer_kwargs))
        self.hidden_layers = nn.ModuleList(layers)

        if readout_type == 'attention':
            self.readout = AttentionReadout(hidden_size, output_size)
        else:
            self.readout = ReadoutLayer(hidden_size, output_size, tau_mem=tau_mem, dt=dt)

    def forward(self, x):
        if self.input_bn_flag:
            B, T, C = x.shape
            x = self.input_bn(x.reshape(B * T, C)).reshape(B, T, C)
        all_spikes = []
        h = x
        for layer in self.hidden_layers:
            spikes, _ = layer(h)
            all_spikes.append(spikes)
            h = spikes
        if self.readout_type == 'attention':
            _, output = self.readout(h)
            out_mem = self.readout.fc(h)
        else:
            out_mem = self.readout(h)
            output, _ = torch.max(out_mem, dim=1)
        return output, all_spikes, out_mem

    def count_params(self):
        return sum(p.numel() for p in self.parameters() if p.requires_grad)


class LSTMBaseline(nn.Module):
    def __init__(self, input_size, hidden_size=128, n_layers=2,
                 output_size=5, dropout=0.3):
        super().__init__()
        self.lstm = nn.LSTM(input_size, hidden_size, n_layers,
                            batch_first=True, dropout=dropout)
        self.fc = nn.Linear(hidden_size, output_size)
    def forward(self, x):
        out, _ = self.lstm(x)
        return self.fc(out[:, -1, :])
    def count_params(self):
        return sum(p.numel() for p in self.parameters() if p.requires_grad)


class CNNBaseline(nn.Module):
    def __init__(self, input_channels, output_size=5):
        super().__init__()
        self.conv = nn.Sequential(
            nn.Conv1d(input_channels, 64, 5, padding=2), nn.BatchNorm1d(64), nn.ReLU(), nn.MaxPool1d(2),
            nn.Conv1d(64, 128, 5, padding=2), nn.BatchNorm1d(128), nn.ReLU(), nn.MaxPool1d(2),
            nn.Conv1d(128, 256, 3, padding=1), nn.BatchNorm1d(256), nn.ReLU(), nn.AdaptiveAvgPool1d(1),
        )
        self.fc = nn.Linear(256, output_size)
    def forward(self, x):
        return self.fc(self.conv(x.transpose(1, 2)).squeeze(-1))
    def count_params(self):
        return sum(p.numel() for p in self.parameters() if p.requires_grad)

print('All model classes loaded.')

All model classes loaded.


## 3. Training Engine

In [7]:
def spike_regularization(all_spikes, theta_l=0.01, s_l=1.0, theta_u=100.0, s_u=0.06):
    reg = torch.tensor(0.0, device=all_spikes[0].device)
    for spk in all_spikes:
        B, T, N = spk.shape
        mean_rate = spk.sum(dim=1) / T
        reg += s_l / (B * N) * (F.relu(theta_l - mean_rate) ** 2).sum()
        pop_count = spk.sum(dim=(1, 2)) / N
        reg += s_u / B * (F.relu(pop_count - theta_u) ** 2).sum()
    return reg


def train_snn(model, train_ld, val_ld, test_ld, n_epochs=80, lr=1e-3,
              device='cuda', patience=20, verbose_every=20):
    model = model.to(device)
    optimizer = torch.optim.Adam(model.parameters(), lr=lr)
    criterion = nn.CrossEntropyLoss()
    best_val, best_state, wait = 0, None, 0
    t0 = time.time()
    for epoch in range(n_epochs):
        model.train()
        correct, total = 0, 0
        for x, y in train_ld:
            x, y = x.to(device), y.to(device)
            logits, spks, _ = model(x)
            loss = criterion(logits, y) + spike_regularization(spks)
            optimizer.zero_grad()
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            optimizer.step()
            correct += (logits.argmax(1) == y).sum().item()
            total += len(y)
        tr_acc = correct / total

        # Validation
        model.eval()
        vc, vt = 0, 0
        with torch.no_grad():
            for x, y in val_ld:
                x, y = x.to(device), y.to(device)
                logits, _, _ = model(x)
                vc += (logits.argmax(1) == y).sum().item()
                vt += len(y)
        va_acc = vc / vt
        if va_acc > best_val:
            best_val = va_acc
            best_state = copy.deepcopy(model.state_dict())
            wait = 0
        else:
            wait += 1
        if wait >= patience:
            break
        if epoch % verbose_every == 0:
            print(f'  Ep {epoch:3d}: tr={tr_acc:.4f} va={va_acc:.4f} best_va={best_val:.4f}')

    if best_state is not None:
        model.load_state_dict(best_state)
    elapsed = time.time() - t0

    # Full test evaluation with F1
    model.eval()
    all_preds, all_labels = [], []
    with torch.no_grad():
        for x, y in test_ld:
            x, y = x.to(device), y.to(device)
            logits, _, _ = model(x)
            all_preds.append(logits.argmax(1).cpu())
            all_labels.append(y.cpu())
    preds = torch.cat(all_preds).numpy()
    labels = torch.cat(all_labels).numpy()
    acc = accuracy_score(labels, preds)
    f1_w = f1_score(labels, preds, average='weighted')
    f1_m = f1_score(labels, preds, average='macro')
    return acc, f1_w, f1_m, model, elapsed, preds, labels


def train_baseline(model, train_ld, val_ld, test_ld, n_epochs=80, lr=1e-3,
                   device='cuda', patience=20):
    model = model.to(device)
    optimizer = torch.optim.Adam(model.parameters(), lr=lr)
    criterion = nn.CrossEntropyLoss()
    best_val, best_state, wait = 0, None, 0
    for epoch in range(n_epochs):
        model.train()
        for x, y in train_ld:
            x, y = x.to(device), y.to(device)
            loss = criterion(model(x), y)
            optimizer.zero_grad(); loss.backward(); optimizer.step()
        model.eval()
        vc, vt = 0, 0
        with torch.no_grad():
            for x, y in val_ld:
                x, y = x.to(device), y.to(device)
                vc += (model(x).argmax(1) == y).sum().item()
                vt += len(y)
        va = vc / vt
        if va > best_val:
            best_val = va; best_state = copy.deepcopy(model.state_dict()); wait = 0
        else:
            wait += 1
        if wait >= patience: break
    if best_state: model.load_state_dict(best_state)
    model.eval()
    all_p, all_l = [], []
    with torch.no_grad():
        for x, y in test_ld:
            x, y = x.to(device), y.to(device)
            all_p.append(model(x).argmax(1).cpu()); all_l.append(y.cpu())
    preds = torch.cat(all_p).numpy(); labels = torch.cat(all_l).numpy()
    acc = accuracy_score(labels, preds)
    f1_w = f1_score(labels, preds, average='weighted')
    f1_m = f1_score(labels, preds, average='macro')
    return acc, f1_w, f1_m, model

print('Training functions ready.')

Training functions ready.


## 4. Architecture Configurations (B0–B7)

Each variant changes exactly one thing from base. Matches iter-7-master notebooks.

| ID | Change | Reference |
|---|---|---|
| B0 | Base RSNN | Cramer et al. 2020 |
| B1 | ALIF (adaptive threshold) | Bellec et al. 2020 |
| B2 | Heterogeneous time constants | Perez-Nieves et al. 2021 |
| B3 | 2-layer RSNN | Cramer et al. 2020 Fig 8b |
| B4 | Input batch normalisation | Kim & Panda 2021 |
| B5 | Attention-weighted readout | Yao et al. 2023 |
| B6 | Scale (512 neurons) | Cramer et al. 2020 |
| B7 | Best combo (top-2 per dataset) | Data-driven |

In [8]:
BASE = dict(
    hidden_size=256, n_hidden_layers=1, recurrent=True,
    tau_mem=20.0, tau_syn=10.0, dt=10.0,
    learnable_tau=True, dropout=0.3,
    neuron_type='lif', heterogeneous_tau=False,
    input_bn=False, readout_type='max_over_time',
)

ARCHS = {
    'B0_base':     {**BASE},
    'B1_alif':     {**BASE, 'neuron_type': 'alif'},
    'B2_hetero':   {**BASE, 'heterogeneous_tau': True},
    'B3_2layer':   {**BASE, 'n_hidden_layers': 2},
    'B4_inputBN':  {**BASE, 'input_bn': True},
    'B5_attention': {**BASE, 'readout_type': 'attention'},
    'B6_scale512': {**BASE, 'hidden_size': 512},
}
# B7 will be constructed per-dataset from top-2 results

print(f'{len(ARCHS)} architectures defined (B7 constructed dynamically).')

7 architectures defined (B7 constructed dynamically).


## 5. Multi-Seed Runner

In [9]:
def run_multiseed(name, config, tr, va, te, in_sz, out_sz, ds_name,
                  seeds=SEEDS, epochs=80, lr=1e-3):
    """Train a config N_SEEDS times, return mean/std of acc, f1_w, f1_m."""
    accs, f1ws, f1ms = [], [], []
    print(f'\n  {name} on {ds_name} ({len(seeds)} seeds)')
    for i, seed in enumerate(seeds):
        set_seed(seed)
        model = SNN(input_size=in_sz, output_size=out_sz, **config)
        acc, f1w, f1m, _, elapsed, preds, labels = train_snn(
            model, tr, va, te, epochs, lr, device, patience=20, verbose_every=999)
        accs.append(acc); f1ws.append(f1w); f1ms.append(f1m)
        print(f'    Seed {seed}: acc={acc*100:.2f}% f1w={f1w*100:.2f}% f1m={f1m*100:.2f}% ({elapsed:.0f}s)')
    result = {
        'acc_mean': np.mean(accs), 'acc_std': np.std(accs),
        'f1w_mean': np.mean(f1ws), 'f1w_std': np.std(f1ws),
        'f1m_mean': np.mean(f1ms), 'f1m_std': np.std(f1ms),
        'all_accs': accs, 'all_f1ws': f1ws, 'all_f1ms': f1ms,
    }
    print(f'    => {result["acc_mean"]*100:.2f}% ± {result["acc_std"]*100:.2f}%  '
          f'F1w={result["f1w_mean"]*100:.2f}% ± {result["f1w_std"]*100:.2f}%  '
          f'F1m={result["f1m_mean"]*100:.2f}% ± {result["f1m_std"]*100:.2f}%')
    return result


def run_baseline_multiseed(ModelClass, model_kwargs, tr, va, te, ds_name,
                           seeds=SEEDS, epochs=80, lr=1e-3):
    accs, f1ws, f1ms = [], [], []
    name = ModelClass.__name__
    print(f'\n  {name} on {ds_name} ({len(seeds)} seeds)')
    for seed in seeds:
        set_seed(seed)
        model = ModelClass(**model_kwargs)
        acc, f1w, f1m, _ = train_baseline(model, tr, va, te, epochs, lr, device, patience=20)
        accs.append(acc); f1ws.append(f1w); f1ms.append(f1m)
        print(f'    Seed {seed}: acc={acc*100:.2f}% f1w={f1w*100:.2f}%')
    result = {
        'acc_mean': np.mean(accs), 'acc_std': np.std(accs),
        'f1w_mean': np.mean(f1ws), 'f1w_std': np.std(f1ws),
        'f1m_mean': np.mean(f1ms), 'f1m_std': np.std(f1ms),
    }
    print(f'    => {result["acc_mean"]*100:.2f}% ± {result["acc_std"]*100:.2f}%')
    return result

print('Multi-seed runner ready.')

Multi-seed runner ready.


---
# Part A: SHD (Spiking Heidelberg Digits)

- 8,156 train / 2,264 test
- 100 time bins × 700 channels
- 20 classes
- Same split as iter-0, iter-1-2, iter-2-3

In [10]:
import h5py

DATA_DIR = None
for root, dirs, files in os.walk('/kaggle/input'):
    if 'shd_train.h5' in files:
        DATA_DIR = root; break
assert DATA_DIR, 'SHD dataset not found. Add shd-snns-dataset.'
print(f'SHD: {DATA_DIR}')

class SHDDataset(Dataset):
    def __init__(self, h5_path, n_time_bins=100, n_channels=700, max_time=1.0,
                 channel_jitter_sigma=0, augment=False):
        self.n_time_bins = n_time_bins
        self.n_channels = n_channels
        self.max_time = max_time
        self.channel_jitter_sigma = channel_jitter_sigma
        self.augment = augment
        self.dt = max_time / n_time_bins
        self.channel_factor = 700 // n_channels
        with h5py.File(h5_path, 'r') as f:
            self.labels = f['labels'][:].astype(np.int64)
            self.spike_times = [f['spikes/times'][i].astype(np.float32) for i in range(len(self.labels))]
            self.spike_units = [f['spikes/units'][i].astype(np.int32) for i in range(len(self.labels))]
        self.n_classes = len(np.unique(self.labels))

    def __len__(self): return len(self.labels)

    def __getitem__(self, idx):
        times = self.spike_times[idx]
        units = self.spike_units[idx].copy()
        if self.augment and self.channel_jitter_sigma > 0:
            jitter = np.random.normal(0, self.channel_jitter_sigma, size=units.shape).astype(np.int32)
            units = np.clip(units + jitter, 0, 699)
        if self.channel_factor > 1:
            units = units // self.channel_factor
        frame = np.zeros((self.n_time_bins, self.n_channels), dtype=np.float32)
        bins = (times / self.max_time * self.n_time_bins).astype(np.int32)
        valid = (bins >= 0) & (bins < self.n_time_bins) & (units >= 0) & (units < self.n_channels)
        np.add.at(frame, (bins[valid], units[valid]), 1)
        return torch.tensor(frame), self.labels[idx]

train_path = os.path.join(DATA_DIR, 'shd_train.h5')
test_path = os.path.join(DATA_DIR, 'shd_test.h5')

# Standard loaders (700 channels, no augmentation) — matches iter-1-2 experiments 1-3
shd_tr_ds = SHDDataset(train_path, 100, 700)
shd_te_ds = SHDDataset(test_path, 100, 700)
shd_tr = DataLoader(shd_tr_ds, 256, True, num_workers=2, pin_memory=True)
shd_va = DataLoader(shd_te_ds, 256, False, num_workers=2, pin_memory=True)  # SHD has no val split; use test
shd_te = DataLoader(shd_te_ds, 256, False, num_workers=2, pin_memory=True)

SHD_IN, SHD_CLS = 700, 20
print(f'SHD loaded: train={len(shd_tr_ds)}, test={len(shd_te_ds)}, classes={SHD_CLS}')

SHD: /kaggle/input/datasets/harshita879/shd-snns-dataset


SHD loaded: train=8156, test=2264, classes=20


### A.1 Multi-Seed B0–B7 on SHD

In [11]:
print('='*70)
print('SHD: ARCHITECTURE SWEEP (5 seeds each)')
print('='*70)

shd_results = {}
for name, cfg in ARCHS.items():
    shd_results[name] = run_multiseed(name, cfg, shd_tr, shd_va, shd_te, SHD_IN, SHD_CLS, 'SHD')

# B7: combine top-2
sorted_shd = sorted(shd_results.items(), key=lambda x: x[1]['acc_mean'], reverse=True)
top2 = [sorted_shd[0][0], sorted_shd[1][0]]
print(f'\nSHD top-2: {top2}')

# Build B7 config by merging the two top configs' non-base fields
b7_cfg = {**BASE}
for t in top2:
    for k, v in ARCHS[t].items():
        if v != BASE.get(k):
            b7_cfg[k] = v
print(f'B7 config: { {k:v for k,v in b7_cfg.items() if v != BASE.get(k)} }')
shd_results['B7_combo'] = run_multiseed('B7_combo', b7_cfg, shd_tr, shd_va, shd_te, SHD_IN, SHD_CLS, 'SHD')

ALL_RESULTS['shd'] = shd_results

SHD: ARCHITECTURE SWEEP (5 seeds each)

  B0_base on SHD (5 seeds)


  Ep   0: tr=0.1327 va=0.1369 best_va=0.1369


    Seed 42: acc=69.17% f1w=68.93% f1m=68.93% (548s)


  Ep   0: tr=0.1334 va=0.1864 best_va=0.1864


    Seed 123: acc=67.84% f1w=66.77% f1m=66.71% (585s)


...

    Seed 456: acc=83.13% f1w=82.87% f1m=82.64% (858s)


  Ep   0: tr=0.0980 va=0.2138 best_va=0.2138


    Seed 789: acc=85.16% f1w=85.02% f1m=84.79% (852s)


  Ep   0: tr=0.1423 va=0.2292 best_va=0.2292


    Seed 1024: acc=82.51% f1w=81.51% f1m=81.34% (610s)
    => 83.20% ± 1.01%  F1w=82.84% ± 1.17%  F1m=82.64% ± 1.16%


### A.2 SHD Baselines (Multi-Seed)

In [12]:
shd_baselines = {}
shd_baselines['LSTM'] = run_baseline_multiseed(
    LSTMBaseline, dict(input_size=700, hidden_size=128, n_layers=2, output_size=20, dropout=0.2),
    shd_tr, shd_va, shd_te, 'SHD')
shd_baselines['CNN'] = run_baseline_multiseed(
    CNNBaseline, dict(input_channels=700, output_size=20),
    shd_tr, shd_va, shd_te, 'SHD')
ALL_RESULTS['shd_baselines'] = shd_baselines


  LSTMBaseline on SHD (5 seeds)


    Seed 42: acc=73.63% f1w=73.90%


    Seed 123: acc=75.09% f1w=74.34%


    Seed 456: acc=76.19% f1w=76.65%


    Seed 789: acc=73.19% f1w=72.89%


...

    Seed 42: acc=87.94% f1w=87.62%


    Seed 123: acc=89.62% f1w=88.47%


    Seed 456: acc=87.81% f1w=87.08%


    Seed 789: acc=87.10% f1w=86.39%


    Seed 1024: acc=91.34% f1w=91.26%
    => 88.76% ± 1.53%


---
# Part B: ECG (PTB-XL)

- Strodthoff split: folds 1–8 train / fold 9 val / fold 10 test
- 250 time bins, delta modulation encoding, threshold=0.1
- 5 classes: NORM, MI, STTC, CD, HYP
- Same split as iter-3-4, v2-rsnn-on-ecg

In [13]:
import wfdb, ast

ecg_csv = glob.glob('/kaggle/input/**/ptbxl_database.csv', recursive=True)
assert ecg_csv, 'PTB-XL not found. Add ptb-xl-dataset by khyeh0719.'
ECG_DIR = os.path.dirname(ecg_csv[0])
print(f'ECG_DIR: {ECG_DIR}')

df = pd.read_csv(os.path.join(ECG_DIR, 'ptbxl_database.csv'))
df.scp_codes = df.scp_codes.apply(ast.literal_eval)
scp_df = pd.read_csv(os.path.join(ECG_DIR, 'scp_statements.csv'), index_col=0)
scp_df = scp_df[scp_df.diagnostic == 1.0]
SUPERCLASSES = ['NORM', 'MI', 'STTC', 'CD', 'HYP']
c2i = {c: i for i, c in enumerate(SUPERCLASSES)}

def get_sc(scp):
    best, lk = None, 0
    for k, v in scp.items():
        if k in scp_df.index:
            sc = scp_df.loc[k].diagnostic_class
            if sc in SUPERCLASSES and v > lk: best, lk = sc, v
    return best

df['sc'] = df.scp_codes.apply(get_sc)
df = df.dropna(subset=['sc'])
df['label'] = df.sc.map(c2i)
df_tr = df[df.strat_fold <= 8]
df_va = df[df.strat_fold == 9]
df_te = df[df.strat_fold == 10]

def load_ecg(subset, data_dir):
    sigs, labs = [], []
    for _, row in subset.iterrows():
        try:
            rec = wfdb.rdrecord(os.path.join(data_dir, row.filename_lr))
            if rec.p_signal is not None and rec.p_signal.shape[0] > 0:
                sigs.append(rec.p_signal.astype(np.float32))
                labs.append(row.label)
        except: pass
    return np.array(sigs), np.array(labs)

print('Loading ECG waveforms...')
X_tr_e, y_tr_e = load_ecg(df_tr, ECG_DIR)
X_va_e, y_va_e = load_ecg(df_va, ECG_DIR)
X_te_e, y_te_e = load_ecg(df_te, ECG_DIR)
print(f'ECG: tr={X_tr_e.shape}, va={X_va_e.shape}, te={X_te_e.shape}')

# Z-score normalise
tr_mean = X_tr_e.mean(axis=(0, 1), keepdims=True)
tr_std = X_tr_e.std(axis=(0, 1), keepdims=True) + 1e-8
X_tr_e = (X_tr_e - tr_mean) / tr_std
X_va_e = (X_va_e - tr_mean) / tr_std
X_te_e = (X_te_e - tr_mean) / tr_std

class ECGDataset(Dataset):
    def __init__(self, X, y, encoding='delta', n_time_bins=250, **kw):
        self.X, self.y = X, torch.tensor(y, dtype=torch.long)
        self.enc, self.n_bins, self.kw = encoding, n_time_bins, kw
        self.input_size = self._enc(X[0]).shape[1]
    def _enc(self, sig):
        idx = np.linspace(0, sig.shape[0]-1, self.n_bins).astype(int)
        s = sig[idx]
        if self.enc == 'direct': return s
        elif self.enc == 'delta': return SpikeEncoder.delta_modulation(s, **self.kw)
        return s
    def __len__(self): return len(self.X)
    def __getitem__(self, i):
        return torch.tensor(self._enc(self.X[i]), dtype=torch.float32), self.y[i]

N_BINS_E, BS_E, N_CLS_E = 250, 128, 5
ecg_tr_ds = ECGDataset(X_tr_e, y_tr_e, 'delta', N_BINS_E, threshold=0.1)
ecg_va_ds = ECGDataset(X_va_e, y_va_e, 'delta', N_BINS_E, threshold=0.1)
ecg_te_ds = ECGDataset(X_te_e, y_te_e, 'delta', N_BINS_E, threshold=0.1)
ECG_IN = ecg_tr_ds.input_size
ecg_tr = DataLoader(ecg_tr_ds, BS_E, True, num_workers=2)
ecg_va = DataLoader(ecg_va_ds, BS_E, False, num_workers=2)
ecg_te = DataLoader(ecg_te_ds, BS_E, False, num_workers=2)

# Direct loaders for baselines
ecg_tr_d = DataLoader(ECGDataset(X_tr_e, y_tr_e, 'direct', N_BINS_E), BS_E, True, num_workers=2)
ecg_va_d = DataLoader(ECGDataset(X_va_e, y_va_e, 'direct', N_BINS_E), BS_E, False, num_workers=2)
ecg_te_d = DataLoader(ECGDataset(X_te_e, y_te_e, 'direct', N_BINS_E), BS_E, False, num_workers=2)

print(f'ECG input: {ECG_IN}, bins: {N_BINS_E}, classes: {N_CLS_E}')

ECG_DIR: /kaggle/input/datasets/khyeh0719/ptb-xl-dataset/ptb-xl-a-large-publicly-available-electrocardiography-dataset-1.0.1


Loading ECG waveforms...


ECG: tr=(17100, 1000, 12), va=(2155, 1000, 12), te=(2162, 1000, 12)


ECG input: 12, bins: 250, classes: 5


### B.1 Multi-Seed B0–B7 on ECG

In [14]:
print('='*70)
print('ECG: ARCHITECTURE SWEEP (5 seeds each)')
print('='*70)

ecg_results = {}
for name, cfg in ARCHS.items():
    ecg_results[name] = run_multiseed(name, cfg, ecg_tr, ecg_va, ecg_te, ECG_IN, N_CLS_E, 'ECG', epochs=100)

sorted_ecg = sorted(ecg_results.items(), key=lambda x: x[1]['acc_mean'], reverse=True)
top2_ecg = [sorted_ecg[0][0], sorted_ecg[1][0]]
print(f'\nECG top-2: {top2_ecg}')
b7_ecg = {**BASE}
for t in top2_ecg:
    for k, v in ARCHS[t].items():
        if v != BASE.get(k): b7_ecg[k] = v
ecg_results['B7_combo'] = run_multiseed('B7_combo', b7_ecg, ecg_tr, ecg_va, ecg_te, ECG_IN, N_CLS_E, 'ECG', epochs=100)
ALL_RESULTS['ecg'] = ecg_results

ECG: ARCHITECTURE SWEEP (5 seeds each)

  B0_base on ECG (5 seeds)


  Ep   0: tr=0.4975 va=0.5360 best_va=0.5360


    Seed 42: acc=62.67% f1w=58.98% f1m=46.81% (5382s)


  Ep   0: tr=0.4989 va=0.5415 best_va=0.5415


    Seed 123: acc=62.67% f1w=59.42% f1m=46.80% (3143s)


  Ep   0: tr=0.5040 va=0.5420 best_va=0.5420


### B.2 ECG Baselines + Per-Class Metrics

In [ ]:
ecg_baselines = {}
ecg_baselines['LSTM'] = run_baseline_multiseed(
    LSTMBaseline, dict(input_size=12, hidden_size=128, n_layers=2, output_size=5, dropout=0.3),
    ecg_tr_d, ecg_va_d, ecg_te_d, 'ECG', epochs=80)
ecg_baselines['CNN'] = run_baseline_multiseed(
    CNNBaseline, dict(input_channels=12, output_size=5),
    ecg_tr_d, ecg_va_d, ecg_te_d, 'ECG', epochs=80)
ALL_RESULTS['ecg_baselines'] = ecg_baselines

# Per-class report for best ECG RSNN (single seed=42 for the confusion matrix)
print('\n--- ECG Per-Class Report (best arch, seed=42) ---')
best_ecg_arch = sorted_ecg[0][0]
set_seed(42)
model_ecg_best = SNN(ECG_IN, output_size=N_CLS_E, **ARCHS[best_ecg_arch] if best_ecg_arch != 'B7_combo' else b7_ecg)
acc, f1w, f1m, _, _, preds_e, labels_e = train_snn(model_ecg_best, ecg_tr, ecg_va, ecg_te, 100, 1e-3, device)
print(classification_report(labels_e, preds_e, target_names=SUPERCLASSES, digits=3))

---
# Part C: FI-2010 (Financial LOB)

- Days 1–7 train (80/20 val), Days 8–9 test (139,488 samples)
- T=100, horizon k=50, 3 classes
- Same split as v3-rsnn-on-fi, v4-rsnn-on-fi, lstm-fi notebooks
- Uses BNTT+LT for RSNN (the architecture that closed the gap from 37.56% to 59.15%)

In [ ]:
import subprocess

print('Downloading FI-2010...')
subprocess.run(['wget', '-q',
    'https://raw.githubusercontent.com/zcakhaa/DeepLOB-Deep-Convolutional-Neural-Networks-for-Limit-Order-Books/master/data/data.zip',
    '-O', '/kaggle/working/data.zip'], check=True)
subprocess.run(['unzip', '-q', '-o', '/kaggle/working/data.zip', '-d', '/kaggle/working/'], check=True)
FI_DIR = '/kaggle/working'
print('Done.')

def prep_x(d): return d[:40,:].T.astype(np.float32)
def get_lab(d): return (d[-5:,:].T.astype(int) - 1)

dec_train = np.loadtxt(os.path.join(FI_DIR, 'Train_Dst_NoAuction_DecPre_CF_7.txt'))
test_files = sorted(glob.glob(os.path.join(FI_DIR, 'Test_Dst_NoAuction*.txt')))
dec_test = np.hstack([np.loadtxt(tf) for tf in test_files])

train_lob, train_label = prep_x(dec_train), get_lab(dec_train)
test_lob, test_label = prep_x(dec_test), get_lab(dec_test)

SEQ_LEN, HORIZON = 100, 3

def make_seq(X, y, sl):
    n = len(X) - sl + 1
    return np.array([X[i:i+sl] for i in range(n)]).astype(np.float32), y[sl-1:]

X_tr_seq, y_tr_seq = make_seq(train_lob, train_label[:, HORIZON], SEQ_LEN)
X_te_seq, y_te_seq = make_seq(test_lob, test_label[:, HORIZON], SEQ_LEN)

val_split = int(len(X_tr_seq) * 0.8)
X_va_f = X_tr_seq[val_split:]; y_va_f = y_tr_seq[val_split:]
X_tr_f = X_tr_seq[:val_split]; y_tr_f = y_tr_seq[:val_split]

class LOBDataset(Dataset):
    def __init__(self, X, y):
        self.X = torch.tensor(X, dtype=torch.float32)
        self.y = torch.tensor(y, dtype=torch.long)
    def __len__(self): return len(self.X)
    def __getitem__(self, i): return self.X[i], self.y[i]

BS_F = 256
fi_tr = DataLoader(LOBDataset(X_tr_f, y_tr_f), BS_F, True, num_workers=2, pin_memory=True, drop_last=True)
fi_va = DataLoader(LOBDataset(X_va_f, y_va_f), BS_F, False, num_workers=2, pin_memory=True)
fi_te = DataLoader(LOBDataset(X_te_seq, y_te_seq), BS_F, False, num_workers=2, pin_memory=True)

FI_IN, FI_CLS = 40, 3
print(f'FI-2010: tr={X_tr_f.shape}, va={X_va_f.shape}, te={X_te_seq.shape}')
print(f'Test labels: {np.bincount(y_te_seq)}')

### C.1 FI-2010 RSNN with BNTT+LT (Multi-Seed)

Since the base RSNN (threshold=1.0) fails on FI-2010 (37.56%), we use the BNTT+LT
architecture from v3. B0–B7 on raw RSNN would all fail for the same threshold reason.
Instead, we run the key comparison: base RSNN (direct) vs BNTT+LT vs baselines.

In [ ]:
# FI-2010 specific: BNTT+LT SNN (from v3-rsnn-on-fi.ipynb)
class LIFLayerBNTT_LT(nn.Module):
    def __init__(self, input_size, hidden_size, n_steps, recurrent=False,
                 tau_mem_init=20.0, tau_syn_init=10.0, dt=10.0,
                 learnable_tau=False, dropout=0.0, threshold_init=0.1):
        super().__init__()
        self.hidden_size = hidden_size
        self.recurrent = recurrent
        self.dt = dt
        self.dropout = dropout
        self.n_steps = n_steps
        self.W_ff = nn.Linear(input_size, hidden_size, bias=False)
        if recurrent:
            self.W_rec = nn.Linear(hidden_size, hidden_size, bias=False)
        self.bn = nn.ModuleList([nn.BatchNorm1d(hidden_size) for _ in range(n_steps)])
        self.log_threshold = nn.Parameter(torch.tensor(np.log(threshold_init)))
        if learnable_tau:
            self.log_tau_mem = nn.Parameter(torch.tensor(np.log(tau_mem_init)))
            self.log_tau_syn = nn.Parameter(torch.tensor(np.log(tau_syn_init)))
        else:
            self.register_buffer('log_tau_mem', torch.tensor(np.log(tau_mem_init)))
            self.register_buffer('log_tau_syn', torch.tensor(np.log(tau_syn_init)))
        nn.init.kaiming_uniform_(self.W_ff.weight, nonlinearity='linear')
        if recurrent:
            nn.init.kaiming_uniform_(self.W_rec.weight, nonlinearity='linear')

    @property
    def alpha(self): return torch.exp(-self.dt / torch.exp(self.log_tau_syn))
    @property
    def beta_decay(self): return torch.exp(-self.dt / torch.exp(self.log_tau_mem))
    @property
    def threshold(self): return torch.exp(self.log_threshold)

    def forward(self, x):
        B, T, _ = x.shape
        alpha, beta, thr = self.alpha, self.beta_decay, self.threshold
        syn = torch.zeros(B, self.hidden_size, device=x.device)
        mem = torch.zeros(B, self.hidden_size, device=x.device)
        prev_spk = torch.zeros(B, self.hidden_size, device=x.device)
        spk_rec, mem_rec = [], []
        for t in range(T):
            cur = self.bn[t](self.W_ff(x[:, t]))
            syn = alpha * syn + cur
            if self.recurrent:
                rec_in = F.dropout(prev_spk, p=self.dropout, training=self.training) if self.dropout > 0 else prev_spk
                syn = syn + self.W_rec(rec_in)
            mem = beta * mem * (1.0 - prev_spk) + (1.0 - beta) * syn
            spk = spike_fn(mem, thr)
            spk_rec.append(spk); mem_rec.append(mem)
            prev_spk = spk
        return torch.stack(spk_rec, dim=1), torch.stack(mem_rec, dim=1)


class SNN_BNTT_LT(nn.Module):
    def __init__(self, input_size=40, hidden_size=256, output_size=3, n_steps=100,
                 recurrent=True, tau_mem=20.0, tau_syn=10.0, dt=10.0,
                 learnable_tau=True, dropout=0.3, threshold_init=0.1):
        super().__init__()
        self.lif = LIFLayerBNTT_LT(input_size, hidden_size, n_steps, recurrent=recurrent,
                                     tau_mem_init=tau_mem, tau_syn_init=tau_syn, dt=dt,
                                     learnable_tau=learnable_tau, dropout=dropout,
                                     threshold_init=threshold_init)
        self.readout = ReadoutLayer(hidden_size, output_size, tau_mem=tau_mem, dt=dt)

    def forward(self, x):
        spikes, _ = self.lif(x)
        out_mem = self.readout(spikes)
        output, _ = torch.max(out_mem, dim=1)
        return output, [spikes], out_mem

    def count_params(self):
        return sum(p.numel() for p in self.parameters() if p.requires_grad)

print(f'BNTT+LT model loaded. Params: {SNN_BNTT_LT().count_params():,}')

In [ ]:
# Multi-seed on FI-2010
print('='*70)
print('FI-2010: MULTI-SEED VARIANCE (5 seeds)')
print('='*70)

fi_results = {}

# Base RSNN (direct input, threshold=1.0) — expected ~37%
fi_results['RSNN_base'] = run_multiseed(
    'RSNN_base', BASE, fi_tr, fi_va, fi_te, FI_IN, FI_CLS, 'FI-2010', epochs=80)

# BNTT+LT — expected ~59%
print('\n--- BNTT+LT (5 seeds) ---')
bntt_accs, bntt_f1ws, bntt_f1ms = [], [], []
for seed in SEEDS:
    set_seed(seed)
    m = SNN_BNTT_LT(FI_IN, 256, FI_CLS, n_steps=SEQ_LEN, threshold_init=0.1)
    # Use Adamax + class weights to match v3
    m = m.to(device)
    counts = torch.bincount(torch.tensor(y_tr_f), minlength=3).float()
    cw = ((1.0 / counts) / (1.0 / counts).sum() * 3).to(device)
    optimizer = torch.optim.Adamax(m.parameters(), lr=1e-3)
    criterion = nn.CrossEntropyLoss(weight=cw)
    best_val, best_state, wait = 0, None, 0
    for ep in range(80):
        m.train()
        for x, y in fi_tr:
            x, y = x.to(device), y.to(device)
            logits, spks, _ = m(x)
            loss = criterion(logits, y) + spike_regularization(spks)
            optimizer.zero_grad(); loss.backward()
            torch.nn.utils.clip_grad_norm_(m.parameters(), 1.0)
            optimizer.step()
        m.eval()
        vc, vt = 0, 0
        with torch.no_grad():
            for x, y in fi_va:
                x, y = x.to(device), y.to(device)
                vc += (m(x)[0].argmax(1) == y).sum().item(); vt += len(y)
        va = vc / vt
        if va > best_val: best_val = va; best_state = copy.deepcopy(m.state_dict()); wait = 0
        else: wait += 1
        if wait >= 20: break
    if best_state: m.load_state_dict(best_state)
    m.eval()
    all_p, all_l = [], []
    with torch.no_grad():
        for x, y in fi_te:
            x, y = x.to(device), y.to(device)
            all_p.append(m(x)[0].argmax(1).cpu()); all_l.append(y.cpu())
    preds = torch.cat(all_p).numpy(); labels = torch.cat(all_l).numpy()
    acc = accuracy_score(labels, preds)
    f1w = f1_score(labels, preds, average='weighted')
    f1m = f1_score(labels, preds, average='macro')
    bntt_accs.append(acc); bntt_f1ws.append(f1w); bntt_f1ms.append(f1m)
    print(f'  Seed {seed}: acc={acc*100:.2f}% f1w={f1w*100:.2f}% f1m={f1m*100:.2f}%')

fi_results['BNTT_LT'] = {
    'acc_mean': np.mean(bntt_accs), 'acc_std': np.std(bntt_accs),
    'f1w_mean': np.mean(bntt_f1ws), 'f1w_std': np.std(bntt_f1ws),
    'f1m_mean': np.mean(bntt_f1ms), 'f1m_std': np.std(bntt_f1ms),
}
print(f'  => BNTT+LT: {np.mean(bntt_accs)*100:.2f}% ± {np.std(bntt_accs)*100:.2f}%')

ALL_RESULTS['fi2010'] = fi_results

### C.2 FI-2010 Baselines + Per-Class Metrics

In [ ]:
fi_baselines = {}
fi_baselines['LSTM'] = run_baseline_multiseed(
    LSTMBaseline, dict(input_size=40, hidden_size=128, n_layers=2, output_size=3, dropout=0.3),
    fi_tr, fi_va, fi_te, 'FI-2010')
fi_baselines['CNN'] = run_baseline_multiseed(
    CNNBaseline, dict(input_channels=40, output_size=3),
    fi_tr, fi_va, fi_te, 'FI-2010')
ALL_RESULTS['fi2010_baselines'] = fi_baselines

# Per-class report for BNTT+LT (seed=42)
print('\n--- FI-2010 Per-Class Report (BNTT+LT, seed=42) ---')
set_seed(42)
m_fi = SNN_BNTT_LT(FI_IN, 256, FI_CLS, n_steps=SEQ_LEN, threshold_init=0.1)
m_fi = m_fi.to(device)
counts = torch.bincount(torch.tensor(y_tr_f), minlength=3).float()
cw = ((1.0 / counts) / (1.0 / counts).sum() * 3).to(device)
opt = torch.optim.Adamax(m_fi.parameters(), lr=1e-3)
crit = nn.CrossEntropyLoss(weight=cw)
best_val, best_state, wait = 0, None, 0
for ep in range(80):
    m_fi.train()
    for x, y in fi_tr:
        x, y = x.to(device), y.to(device)
        lo, sp, _ = m_fi(x)
        loss = crit(lo, y) + spike_regularization(sp)
        opt.zero_grad(); loss.backward()
        torch.nn.utils.clip_grad_norm_(m_fi.parameters(), 1.0); opt.step()
    m_fi.eval()
    vc, vt = 0, 0
    with torch.no_grad():
        for x, y in fi_va:
            x, y = x.to(device), y.to(device)
            vc += (m_fi(x)[0].argmax(1)==y).sum().item(); vt += len(y)
    va = vc/vt
    if va > best_val: best_val=va; best_state=copy.deepcopy(m_fi.state_dict()); wait=0
    else: wait+=1
    if wait>=20: break
if best_state: m_fi.load_state_dict(best_state)
m_fi.eval()
all_p, all_l = [], []
with torch.no_grad():
    for x, y in fi_te:
        x, y = x.to(device), y.to(device)
        all_p.append(m_fi(x)[0].argmax(1).cpu()); all_l.append(y.cpu())
preds_f = torch.cat(all_p).numpy(); labels_f = torch.cat(all_l).numpy()
print(classification_report(labels_f, preds_f, target_names=['Down','Stationary','Up'], digits=3))

---
## Final Cross-Dataset Summary

In [ ]:
print('='*80)
print('COMPLETE RESULTS: MULTI-SEED ARCHITECTURE COMPARISON')
print('='*80)

for ds_name, ds_key, bl_key in [
    ('SHD', 'shd', 'shd_baselines'),
    ('ECG', 'ecg', 'ecg_baselines'),
]:
    print(f'\n--- {ds_name} ---')
    print(f'{"Architecture":<20} {"Acc (mean±std)":>20} {"F1w (mean±std)":>20} {"F1m (mean±std)":>20}')
    print('-'*82)
    for name, r in sorted(ALL_RESULTS[ds_key].items(), key=lambda x: x[1]['acc_mean'], reverse=True):
        print(f'  {name:<18} {r["acc_mean"]*100:>6.2f}% ± {r["acc_std"]*100:.2f}%  '
              f'{r["f1w_mean"]*100:>6.2f}% ± {r["f1w_std"]*100:.2f}%  '
              f'{r["f1m_mean"]*100:>6.2f}% ± {r["f1m_std"]*100:.2f}%')
    for name, r in ALL_RESULTS[bl_key].items():
        print(f'  {name:<18} {r["acc_mean"]*100:>6.2f}% ± {r["acc_std"]*100:.2f}%  '
              f'{r["f1w_mean"]*100:>6.2f}% ± {r["f1w_std"]*100:.2f}%  '
              f'{r["f1m_mean"]*100:>6.2f}% ± {r["f1m_std"]*100:.2f}%')

print(f'\n--- FI-2010 ---')
print(f'{"Model":<20} {"Acc (mean±std)":>20} {"F1w (mean±std)":>20} {"F1m (mean±std)":>20}')
print('-'*82)
for name, r in sorted(ALL_RESULTS['fi2010'].items(), key=lambda x: x[1]['acc_mean'], reverse=True):
    print(f'  {name:<18} {r["acc_mean"]*100:>6.2f}% ± {r["acc_std"]*100:.2f}%  '
          f'{r["f1w_mean"]*100:>6.2f}% ± {r["f1w_std"]*100:.2f}%  '
          f'{r["f1m_mean"]*100:>6.2f}% ± {r["f1m_std"]*100:.2f}%')
for name, r in ALL_RESULTS['fi2010_baselines'].items():
    print(f'  {name:<18} {r["acc_mean"]*100:>6.2f}% ± {r["acc_std"]*100:.2f}%  '
          f'{r["f1w_mean"]*100:>6.2f}% ± {r["f1w_std"]*100:.2f}%  '
          f'{r["f1m_mean"]*100:>6.2f}% ± {r["f1m_std"]*100:.2f}%')

In [ ]:
# Save all results
with open(os.path.join(SAVE_DIR, 'final_multiseed_results.json'), 'w') as f:
    # Convert numpy to float for JSON
    def convert(obj):
        if isinstance(obj, np.floating): return float(obj)
        if isinstance(obj, np.ndarray): return obj.tolist()
        if isinstance(obj, dict): return {k: convert(v) for k, v in obj.items()}
        return obj
    json.dump(convert(ALL_RESULTS), f, indent=2)
print('Saved to final_multiseed_results.json')